# 6.2. Parameter Management
D2L의 Parameter Management장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [2]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 파라미터(parameter)란?

신경망을 학습한다는 것은 결국 모델 내부의 파라미터를 적절한 값으로 바꾸는 과정이다.

대표 파라미터는 2가지가 있다.

- `weight`: 가중치 $w$
- `bias`: 편향 $b$

예를 들어서 선형층이 아래처럼 계산을 할때

$$
Y = XW^T + b
$$

    nn.Linear(4, 8)
을 하면 입력 feature 수는 4개, 출력 뉴런 수는 8개이다. 내부에는 학습해야 할 weight와 bias가 자동으로 만들어진다.

모델 학습 -> loss 계산 -> backward -> weight, bias의 gadient 계산 -> optimizer가 weight, bias 수정

D2L에서도 학습의 목적을 손실을 줄이느 파라미터 값을 찾는 것으로 설명하고, 학습 후 이 파라미터를 예측, 저장, 분석 등에 사용할 수 있다고 설명한다.

## 2. 실습용 신경망

간단한 MLP를 만들어 보겠다. 구조는 이렇다.

```text
입력
[batch_size, 4]

    ↓ Linear(4, 8)  // 4개의 feature받아 8개 값 만듬.

[batch_size, 8]

    ↓ ReLU

[batch_size, 8]

    ↓ Linear(8, 1)  // 8개의 값을 받아 최종적으로 1개의 값을 만듬.

출력
[batch_size, 1]
```

In [3]:
net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

X = torch.rand(2, 4)

y_hat = net(X)

print(y_hat.shape)

torch.Size([2, 1])


## 3. Sequential 안의 층 접근

`nn.Sequential`은 여러 층을 순서대로 보관한다. 따라서 Python 리스트처럼 번호로 층에 접근할 수 있다.

```text
net[0] → Linear(4, 8)
net[1] → ReLU()
net[2] → Linear(8, 1)
```

여기서 ReLU는 학습해야할 weight나 bias가 없다. Linear는 weight, bias가 있다.

In [4]:
print(net)

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)


## 4. 한 층의 파라미터 확인

Linear layer는 기본적으로 두개의 파라미터가 있다. (w, b)

`state_dict()`를 사용하면 해당 층이 가지고 있는 값고 이름을 확인할 수 있다.

In [5]:
net[2].state_dict()

OrderedDict([('weight',
              tensor([[ 0.0950, -0.0959,  0.1488,  0.3157,  0.2044, -0.1546,  0.2041,  0.0633]])),
             ('bias', tensor([0.1795]))])

## 5. Weight와 Bias의 Shape

현재 첫째 층은 

    nn.Linear(4, 8)

이다. 입력 feature가 4개이고 출력 뉴런이 8개라서

```text
weight.shape = [8, 4]
bias.shape   = [8]
```

8개의 출력 뉴런 각각이 입력 feature 4개에 대한 weight를 가져야 하기 때문이다.

두 번째 층은

    nn.Linear(8, 1)

```text
weight.shape = [1, 8]
bias.shape   = [1]
```

In [6]:
print(net[0].weight.shape)
print(net[0].bias.shape)

print(net[2].weight.shape)
print(net[2].bias.shape)

torch.Size([8, 4])
torch.Size([8])
torch.Size([1, 8])
torch.Size([1])


## 6. 특정 파라미터 직접 접근

Linear layer의 weight와 bias에 직접 접근할 수 있다.

```python
layer.weight
layer.bias

net[2].weight
net[2].bias
```

이 값들은 그냥 Tensor가 아니라 `nn.Parameter`객체이다.

In [7]:
print(type(net[2].weight))
print(type(net[2].bias))

<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>


`nn.Parameter`는 PyTorch에게 학습해야 하는 값이라고 알려주는 역할을 한다. 그래서 optimizer가 모델의 parameter들을 찾아서 업데이트할 수 있다.

D2L도 모델 파라미터가 단순한 숫자 배열이 아니라 값과 gradient 등의 정보를 담는 파라미터 객체라는 점을 강조합니다.

## 7. Parameter에는 값과 gradient가 있다.

Parameter는 단순히 숫자만 가지고 있지 않는다.

```text
Parameter -> 현재 weight 값 , gradient
```

In [ ]:
print(net[2].weight)

print(net[2].weight.detach()) # detach()를 사용하면 autograd 계산 그래프와 분리된 값으로 확인가능

print(net[2].weight.grad) # backward()를 하지않아 grad가 없다

Parameter containing:
tensor([[ 0.0950, -0.0959,  0.1488,  0.3157,  0.2044, -0.1546,  0.2041,  0.0633]],
       requires_grad=True)
tensor([[ 0.0950, -0.0959,  0.1488,  0.3157,  0.2044, -0.1546,  0.2041,  0.0633]])
None


## 8. Backward 후 Gradient 확인하기

순전파만 수행하면 gradient는 아직 계산되지 않는다. 역전파를 해야 gradient가 만들어지는데 흐름은 이렇다.

```text
X
↓
모델
↓
y_hat
↓
loss 계산
↓
loss.backward()
↓
각 Parameter의 .grad 계산
```

In [11]:
X = torch.rand(2, 4)
y = torch.rand(2, 1)

y_hat = net(X)

loss_fn = nn.MSELoss()
loss = loss_fn(y_hat, y)

loss.backward()

In [ ]:
print(net[2].weight.grad) # None이었는데 이제 숫자가 나온다.
print(net[2].bias.grad)

tensor([[-0.4774, -0.1219,  0.0000, -0.2157, -0.0864, -0.2448,  0.0000,  0.0000]])
tensor([-0.4990])


weight = 현재 가중치이고

weight.grad
= loss 가 이 weight에 대해 얼마나 변하는지
= optimizer가 weight를 어느 방향으로 수정해야 하는지 알려줌

## 9. 모델의 모든 파라미터 확인하기

신경망이 커지만 하나씩 확인하는 것은 불편하다.

PyTorch에서는 `named_parameters()`를 사용해 모델 전체의 파라미터를 순회할 수 있다.

In [ ]:
for name, param in net.named_parameters():
    print(name, param.shape) # 1번은 ReLU라 없음

0.weight torch.Size([8, 4])
0.bias torch.Size([8])
2.weight torch.Size([1, 8])
2.bias torch.Size([1])


## 10. Parameters()와 named_parameters()

모델의 parameter를 가져오는 대표적인 방법은 두 가지이다.

### Parameters()

parameter 값만 가져옴.

In [14]:
for param in net.parameters():
    print(param.shape)

torch.Size([8, 4])
torch.Size([8])
torch.Size([1, 8])
torch.Size([1])


### named_parameters()

parameter의 이름과 값을 같이 가져온다.

In [15]:
for name, param in net.named_parameters():
    print(name, param.shape)

0.weight torch.Size([8, 4])
0.bias torch.Size([8])
2.weight torch.Size([1, 8])
2.bias torch.Size([1])


이제 이전에 optimizer 코드도 해석할 수 있다.

In [ ]:
optimizer = torch.optim.SGD(
    net.parameters(), # 이 모델 안 w, b를 업데이트 해라
    lr=0.01
)

## 11. 파라미터 공유(Parameter Sharing)

서로 다른 위치에서 같은 weight를 사용하도록 만들 수도 있다. 이것이 파라미터 공유이다. 예를 들어서

In [16]:
shared = nn.Linear(8, 8)

In [ ]:
net_shared = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),

    shared, # 1
    nn.ReLU(),

    shared, # 2
    nn.ReLU(),

    nn.Linear(8, 1)
)

## 12. 공유된 Parameter의 Gradient는 어떻게 되나

같은 Parameter를 여러 위치에서 사용하면 순전파 과정에서도 같은 weight가 여러 번 사용된다.  

Backward에서는 각 위치에서 해당 weight가 loss에 끼친 영향이 계산되고 그 gradient들이 동일한 Parameter의 `.grad`에 누적된다.

```text
첫 번째 gradient + 두 번째 gradient = shared.weight.grad
```

In [18]:
X = torch.rand(2, 4)
y = torch.rand(2, 1)

y_hat = net_shared(X)

loss = nn.MSELoss()(y_hat, y)

net_shared.zero_grad()
loss.backward()

print(shared.weight.grad.shape)

torch.Size([8, 8])


## 13. Parameter Management 전체 흐름

In [19]:
net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

모델을 만들면 내부에 자동으로 Parameter가 만들어진다.

```text
net
│
├── Linear(4, 8)
│   ├── weight [8, 4]
│   └── bias   [8]
│
├── ReLU
│   └── parameter 없음
│
└── Linear(8, 1)
    ├── weight [1, 8]
    └── bias   [1]
```

신경망 학습은 이것의 반복이다.
```text
Parameter 사용
-> prediction
-> loss
-> backward
-> Parameter의 gradient 계산
-> optimizer가 Parameter 수정
```

## 14. 오늘의 정리

- 신경망의 `weight`와 `bias`는 학습되는 Parameter이다.
- Sequential에서는 net[0], net[2]처럼 특정 층에 접근할 수 있다.
- Parameter는 단순한 숫자가 아니라 학습 대상인 Tensor이며 gradient도 관리한다.
- optimizer에 net.parameters()를 전달하는 이유는 모델의 weight와 bias를 업데이트하기 위해서다.
- 하나의 Layer 객체를 여러 위치에서 사용하면 parameter sharing이 일어난다.
- 공유된 parameter는 값만 같은 것이 아니라 실제로 동일한 Parameter 객체이다.
- 공유된 parameter가 여러 위치에서 사용되면 역전파 과정에서 각 사용 위치에서 발생한 gradient가 같은 parameter에 누적된다.